# 2D U(1) Large-Angle Edge-Link Weighting

This notebook reports the 2du1 experiment that up-weights gauge links whose angles sit near $\pm\pi$. Topology tunneling is mainly due to those large-angle links, so the training force loss concentrates on them. The architecture remains production `base`. Only the PyTorch objective changes. JAX FT-HMC evaluation is unchanged.

**Conclusion.** On L=16, train $\beta=3.0$, eight seeds, and 2048-config FT-HMC, the extra weight does not produce a useful topology gain over production `base`. Smooth settings stay within a few percent of `base`. The T1 leader `edge_h15_w3` trades a small $R_\gamma$ excess for a significant $R_{\Delta Q}$ loss. Default training stays unweighted (`edge_weight=0`).

Dumps live under `2du1/evaluation/{hmc,base,edge}/dumps`. Protocol notes are in [`docs/edge_weight.md`](docs/edge_weight.md).

In [1]:
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from scipy.integrate import quad

cwd = Path.cwd()
REPO_ROOT = cwd if (cwd / "2du1").exists() else cwd.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from nthmc.core.plot_settings import color_ls, default_plot, errorb, fs_p, fs_small_p, ls_p

LATTICE_SIZE = 16
TRAIN_BETA = 3.0
EVAL_BETA = 3.0
N_STEPS = 10
MAX_LAG = 20
GAMMA_LAG = 16
SEEDS = [1029, 1107, 1331, 1984, 1999, 2008, 2017, 2025]
VOLUME = LATTICE_SIZE ** 2

TAGS = {
    "base": {"label": r"production base", "form": "none"},
    "edge_s2_w1": {"label": r"smooth $p=2$, $\alpha=1$", "form": "smooth"},
    "edge_s2_w3": {"label": r"smooth $p=2$, $\alpha=3$", "form": "smooth"},
    "edge_s2_w10": {"label": r"smooth $p=2$, $\alpha=10$", "form": "smooth"},
    "edge_h05_w3": {"label": r"hard $\delta=0.05$, $\alpha=3$", "form": "hard"},
    "edge_h15_w3": {"label": r"hard $\delta=0.15$, $\alpha=3$", "form": "hard"},
}
EDGE_TAGS = [tag for tag in TAGS if tag != "base"]

HMC_DIR = REPO_ROOT / "2du1" / "evaluation" / "hmc" / "dumps"
BASE_DIR = REPO_ROOT / "2du1" / "evaluation" / "base" / "dumps"
EDGE_DIR = REPO_ROOT / "2du1" / "evaluation" / "edge" / "dumps"
PLOT_DIR = REPO_ROOT / "presentation" / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

## Objective and protocol

Physical $\theta$ is inverted to $z=F^{-1}(\theta)$. Per-link weights from **physical** $\theta$ multiply the matching $z$-space force:

$$
w(\theta)=1+\alpha\left(\frac{1-\cos\theta}{2}\right)^p
\qquad\text{(smooth, $\delta=0$)}
$$

$$
w(\theta)=1+\alpha\ \text{if}\ |\mathrm{wrap}(\theta)|\ge\pi-\delta,\ \text{else }1
\qquad\text{(hard, $\delta>0$)}
$$

$\alpha=0$ recovers the production force-only loss. Weights are detached. Checkpointing follows the weighted validation objective; the unweighted force loss is the promotion gate.

Train at L=16, $\beta=3.0$, same 4-GPU protocol as production `base`. Score FT-HMC with $R_\gamma(16)=(1-A_{\mathrm{FT}}(16))/(1-A_{\mathrm{ref}}(16))$ and $R_{\Delta Q}=\langle|\Delta Q|\rangle_{\mathrm{FT}}/\langle|\Delta Q|\rangle_{\mathrm{ref}}$. Autocorrelations use the same $\chi_\infty$ formula as `u1_observables.autocorrelation_from_chi`, inlined here so this notebook does not import JAX.

Win rule at $\beta=3$: both metric means beat `base`, and at least one is $1\sigma$ separated. A few-percent excess that is consistent with seed scatter is not treated as a topology gain.

In [2]:
theta = np.linspace(-math.pi, math.pi, 401)
fig, ax = default_plot()
ax.plot(theta, 1 + 1.0 * ((1 - np.cos(theta)) / 2) ** 2, color=color_ls[0], label=r"$p=2$, $\alpha=1$")
ax.plot(theta, 1 + 3.0 * ((1 - np.cos(theta)) / 2) ** 2, color=color_ls[1], label=r"$p=2$, $\alpha=3$")
ax.plot(theta, 1 + 10.0 * ((1 - np.cos(theta)) / 2) ** 2, color=color_ls[2], label=r"$p=2$, $\alpha=10$")
ax.plot(
    theta,
    np.where(np.abs(theta) >= math.pi - 0.05, 4.0, 1.0),
    color=color_ls[3],
    ls="--",
    label=r"hard $\delta=0.05$, $\alpha=3$",
)
ax.plot(
    theta,
    np.where(np.abs(theta) >= math.pi - 0.15, 4.0, 1.0),
    color=color_ls[4],
    ls=":",
    label=r"hard $\delta=0.15$, $\alpha=3$",
)
ax.set_xlabel(r"$\theta$", **fs_p)
ax.set_ylabel(r"$w(\theta)$", **fs_p)
ax.set_xlim(-math.pi, math.pi)
ax.legend(frameon=False, **fs_small_p)
fig.savefig(PLOT_DIR / "edge_weight_w_theta.pdf", bbox_inches="tight")
fig

In [3]:
def chi_infinity(beta):
    numerator, _ = quad(
        lambda phi: (phi / (2 * math.pi)) ** 2 * math.exp(beta * math.cos(phi)),
        -math.pi,
        math.pi,
    )
    denominator, _ = quad(lambda phi: math.exp(beta * math.cos(phi)), -math.pi, math.pi)
    return numerator / denominator


CHI = {3.0: chi_infinity(3.0), 5.0: chi_infinity(5.0)}


def load_vector(path):
    if not path.exists():
        raise FileNotFoundError(path)
    return np.atleast_1d(np.loadtxt(path))


def load_scalar(path):
    return float(load_vector(path).reshape(-1)[0])


def delta_q(topology):
    if len(topology) < 2:
        return float("nan")
    return float(np.mean(np.abs(np.diff(topology))))


def autocorrelation_from_chi(topo, max_lag, beta, volume):
    topo = np.round(topo).astype(float)
    topo = topo - np.mean(topo)
    chi = CHI[float(beta)]
    ac = np.zeros(max_lag + 1)
    ac[0] = 1.0
    for lag in range(1, max_lag + 1):
        if lag >= len(topo):
            ac[lag] = np.nan
        else:
            ac[lag] = 1.0 - np.mean((topo[:-lag] - topo[lag:]) ** 2) / (2 * volume * chi)
    return ac


def jackknife_mean(values):
    values = np.asarray(values, dtype=float)
    n = len(values)
    mean = float(np.mean(values))
    if n < 2:
        return mean, 0.0
    leave = np.array([(n * mean - values[i]) / (n - 1) for i in range(n)])
    sdev = float(np.sqrt((n - 1) / n * np.sum((leave - mean) ** 2)))
    return mean, sdev


def fmt(mean, sdev=None):
    if sdev is None:
        mean, sdev = mean
    return f"{mean:.3f}({round(sdev * 1000):d})"


def sigma_from_one(mean, sdev):
    if sdev <= 0:
        return float("nan")
    return (mean - 1.0) / sdev


def show_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(item) for item in row) + " |")
    display(Markdown("\n".join(lines)))


def topo_paths(tag, beta, seed):
    beta_tag = f"{float(beta):.1f}"
    prefix = f"L{LATTICE_SIZE}_beta{beta_tag}_nsteps{N_STEPS}"
    if tag == "base":
        save = f"base_train_b{TRAIN_BETA:.1f}_L{LATTICE_SIZE}_{seed}"
        dump_dir = BASE_DIR
        kind = "fthmc"
    elif tag == "hmc":
        return {
            "topo": HMC_DIR / f"topo_hmc_{prefix}_{seed}.csv",
            "accept": HMC_DIR / f"accept_rate_hmc_{prefix}_{seed}.csv",
            "step": HMC_DIR / f"step_size_hmc_{prefix}_{seed}.csv",
        }
    else:
        save = f"{tag}_train_b{TRAIN_BETA:.1f}_L{LATTICE_SIZE}_{seed}"
        dump_dir = EDGE_DIR
        kind = "fthmc"
    return {
        "topo": dump_dir / f"topo_{kind}_{prefix}_{save}.csv",
        "accept": dump_dir / f"accept_rate_{kind}_{prefix}_{save}.csv",
        "step": dump_dir / f"step_size_{kind}_{prefix}_{save}.csv",
    }

In [4]:
def collect_tag(tag, beta=EVAL_BETA, seeds=SEEDS, n_keep=None):
    per_seed = []
    for seed in seeds:
        hmc = load_vector(topo_paths("hmc", beta, seed)["topo"])
        base = load_vector(topo_paths("base", beta, seed)["topo"])
        edge = load_vector(topo_paths(tag, beta, seed)["topo"])
        keep = min(len(hmc), len(base), len(edge))
        if n_keep is not None:
            keep = min(keep, int(n_keep))
        hmc, base, edge = hmc[:keep], base[:keep], edge[:keep]
        ae = autocorrelation_from_chi(edge, MAX_LAG, beta, VOLUME)
        ab = autocorrelation_from_chi(base, MAX_LAG, beta, VOLUME)
        ah = autocorrelation_from_chi(hmc, MAX_LAG, beta, VOLUME)
        de, db, dh = delta_q(edge), delta_q(base), delta_q(hmc)
        per_seed.append(
            {
                "seed": seed,
                "n": keep,
                "accept": load_scalar(topo_paths(tag, beta, seed)["accept"]),
                "step": load_scalar(topo_paths(tag, beta, seed)["step"]),
                "r_gamma_vs_base": (1.0 - ae[GAMMA_LAG]) / (1.0 - ab[GAMMA_LAG]),
                "r_deltaq_vs_base": de / db,
                "r_gamma_vs_hmc": (1.0 - ae[GAMMA_LAG]) / (1.0 - ah[GAMMA_LAG]),
                "r_deltaq_vs_hmc": de / dh,
                "r_gamma_base_vs_hmc": (1.0 - ab[GAMMA_LAG]) / (1.0 - ah[GAMMA_LAG]),
                "r_deltaq_base_vs_hmc": db / dh,
                "topo": edge,
            }
        )
    summary = {"tag": tag, "beta": beta, "per_seed": per_seed}
    for key in [
        "r_gamma_vs_base",
        "r_deltaq_vs_base",
        "r_gamma_vs_hmc",
        "r_deltaq_vs_hmc",
        "r_gamma_base_vs_hmc",
        "r_deltaq_base_vs_hmc",
    ]:
        mean, sdev = jackknife_mean([row[key] for row in per_seed])
        summary[key] = {"mean": mean, "sdev": sdev, "sigma": sigma_from_one(mean, sdev)}
    return summary


summaries = {tag: collect_tag(tag) for tag in EDGE_TAGS}
base_vs_hmc_gamma = jackknife_mean(
    [row["r_gamma_base_vs_hmc"] for row in summaries["edge_s2_w1"]["per_seed"]]
)
base_vs_hmc_delta = jackknife_mean(
    [row["r_deltaq_base_vs_hmc"] for row in summaries["edge_s2_w1"]["per_seed"]]
)
n_samples = {tag: [row["n"] for row in summaries[tag]["per_seed"]] for tag in EDGE_TAGS}
assert all(n == 2048 for counts in n_samples.values() for n in counts)

## Eight-seed FT-HMC at $\beta=3.0$ (2048 configs)

Jackknife is over the eight per-seed ratios versus matched HMC and production `base`. All chains have 2048 samples and acceptance in $[0.55,0.85]$ at step $0.35$.

In [5]:
rows = [
    [
        "base",
        fmt(base_vs_hmc_gamma),
        fmt(base_vs_hmc_delta),
        "—",
        "—",
        "—",
        "—",
    ]
]
for tag in EDGE_TAGS:
    s = summaries[tag]
    rows.append(
        [
            f"`{tag}`",
            fmt(s["r_gamma_vs_hmc"]["mean"], s["r_gamma_vs_hmc"]["sdev"]),
            fmt(s["r_deltaq_vs_hmc"]["mean"], s["r_deltaq_vs_hmc"]["sdev"]),
            fmt(s["r_gamma_vs_base"]["mean"], s["r_gamma_vs_base"]["sdev"]),
            fmt(s["r_deltaq_vs_base"]["mean"], s["r_deltaq_vs_base"]["sdev"]),
            f"{s['r_gamma_vs_base']['sigma']:+.1f}$\\sigma$",
            f"{s['r_deltaq_vs_base']['sigma']:+.1f}$\\sigma$",
        ]
    )
show_table(
    ["tag", r"$R_\gamma$ vs HMC", r"$R_{\Delta Q}$ vs HMC", r"$R_\gamma$ vs base", r"$R_{\Delta Q}$ vs base", r"$R_\gamma$ $\sigma$", r"$R_{\Delta Q}$ $\sigma$"],
    rows,
)

acc_rows = []
for tag in EDGE_TAGS:
    acc = [row["accept"] for row in summaries[tag]["per_seed"]]
    step = [row["step"] for row in summaries[tag]["per_seed"]]
    acc_rows.append([f"`{tag}`", f"{min(acc):.3f}–{max(acc):.3f}", f"{min(step):.3f}–{max(step):.3f}"])
show_table(["tag", "accept", "step"], acc_rows)

| tag | $R_\gamma$ vs HMC | $R_{\Delta Q}$ vs HMC | $R_\gamma$ vs base | $R_{\Delta Q}$ vs base | $R_\gamma$ $\sigma$ | $R_{\Delta Q}$ $\sigma$ |
| --- | --- | --- | --- | --- | --- | --- |
| base | 1.073(24) | 1.511(13) | — | — | — | — |
| `edge_s2_w1` | 1.076(28) | 1.536(16) | 1.003(10) | 1.017(10) | +0.3$\sigma$ | +1.7$\sigma$ |
| `edge_s2_w3` | 1.065(41) | 1.519(14) | 0.991(21) | 1.005(9) | -0.5$\sigma$ | +0.6$\sigma$ |
| `edge_s2_w10` | 1.083(40) | 1.544(20) | 1.009(28) | 1.022(12) | +0.3$\sigma$ | +1.9$\sigma$ |
| `edge_h05_w3` | 1.068(21) | 1.419(17) | 0.998(27) | 0.939(9) | -0.1$\sigma$ | -6.7$\sigma$ |
| `edge_h15_w3` | 1.113(19) | 1.453(18) | 1.039(15) | 0.962(11) | +2.6$\sigma$ | -3.5$\sigma$ |


| tag | accept | step |
| --- | --- | --- |
| `edge_s2_w1` | 0.809–0.833 | 0.350–0.350 |
| `edge_s2_w3` | 0.801–0.832 | 0.350–0.350 |
| `edge_s2_w10` | 0.791–0.838 | 0.350–0.350 |
| `edge_h05_w3` | 0.712–0.750 | 0.350–0.350 |
| `edge_h15_w3` | 0.736–0.778 | 0.350–0.350 |


In [6]:
fig, ax = default_plot()
xs = np.arange(len(EDGE_TAGS))
gamma_means = np.array([summaries[tag]["r_gamma_vs_base"]["mean"] for tag in EDGE_TAGS])
gamma_sdev = np.array([summaries[tag]["r_gamma_vs_base"]["sdev"] for tag in EDGE_TAGS])
delta_means = np.array([summaries[tag]["r_deltaq_vs_base"]["mean"] for tag in EDGE_TAGS])
delta_sdev = np.array([summaries[tag]["r_deltaq_vs_base"]["sdev"] for tag in EDGE_TAGS])
ax.errorbar(xs - 0.08, gamma_means, yerr=gamma_sdev, color=color_ls[0], marker="o", label=r"$R_\gamma(16)$ vs base", **errorb)
ax.errorbar(xs + 0.08, delta_means, yerr=delta_sdev, color=color_ls[1], marker="s", label=r"$R_{\Delta Q}$ vs base", **errorb)
ax.axhline(1.0, color="0.4", linestyle="--", linewidth=1)
ax.set_xticks(xs)
ax.set_xticklabels([tag.replace("edge_", "") for tag in EDGE_TAGS], rotation=25, ha="right")
ax.set_ylabel("ratio vs production base", **fs_p)
ax.legend(frameon=False, **fs_small_p)
ax.set_ylim(0.88, 1.12)
fig.savefig(PLOT_DIR / "edge_weight_r_vs_base.pdf", bbox_inches="tight")
fig

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.6))
markers = ["o", "s", "^", "D", "P"]
xs = np.arange(len(SEEDS))
offsets = (np.arange(len(EDGE_TAGS)) - (len(EDGE_TAGS) - 1) / 2) * 0.12
for ax, key, ylabel in zip(
    axes,
    ["r_gamma_vs_base", "r_deltaq_vs_base"],
    [r"$R_\gamma(16)$ vs base", r"$R_{\Delta Q}$ vs base"],
):
    ax.tick_params(direction="in", top="on", right="on", **ls_p)
    ax.grid(linestyle=":")
    for color, marker, tag, dx in zip(color_ls, markers, EDGE_TAGS, offsets):
        ys = [row[key] for row in summaries[tag]["per_seed"]]
        ax.scatter(xs + dx, ys, color=color, marker=marker, s=36, label=tag.replace("edge_", ""))
    ax.axhline(1.0, color="0.4", linestyle="--", linewidth=1)
    ax.set_xlabel("seed", **fs_small_p)
    ax.set_ylabel(ylabel, **fs_small_p)
    ax.set_xticks(xs)
    ax.set_xticklabels([str(seed) for seed in SEEDS])
    ax.set_xlim(-0.6, len(SEEDS) - 0.4)
axes[0].legend(frameon=False, fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(PLOT_DIR / "edge_weight_per_seed.pdf", bbox_inches="tight")
fig

Seed-to-seed scatter is a few percent. Relative to HMC, everyone including production `base` is still around $R_\gamma\sim 1.07$ and $R_{\Delta Q}\sim 1.5$; that is the existing `base` gain, not extra from weighting.

The published T1 512-config screen (a separate short chain, seed 1029) showed $R_\gamma$ swings of $+11\%$ to $+43\%$ versus `base`. The first 512 samples of the later 2048-config chains are also noisy and do not match those T1 numbers or the eight-seed means.

In [8]:
t1_published = {
    "edge_s2_w1": (1.113, 1.156),
    "edge_s2_w3": (1.218, 1.119),
    "edge_s2_w10": (0.999, 1.127),
    "edge_h05_w3": (1.130, 1.037),
    "edge_h15_w3": (1.427, 1.110),
}
t1_trunc = {tag: collect_tag(tag, seeds=[1029], n_keep=512)["per_seed"][0] for tag in EDGE_TAGS}
rows = []
for tag in EDGE_TAGS:
    s = summaries[tag]
    trunc = t1_trunc[tag]
    pub_g, pub_d = t1_published[tag]
    rows.append(
        [
            f"`{tag}`",
            f"{pub_g:.3f}",
            f"{pub_d:.3f}",
            f"{trunc['r_gamma_vs_base']:.3f}",
            f"{trunc['r_deltaq_vs_base']:.3f}",
            fmt(s["r_gamma_vs_base"]["mean"], s["r_gamma_vs_base"]["sdev"]),
            fmt(s["r_deltaq_vs_base"]["mean"], s["r_deltaq_vs_base"]["sdev"]),
        ]
    )
show_table(
    [
        "tag",
        r"T1 $R_\gamma$",
        r"T1 $R_{\Delta Q}$",
        r"512 trunc $R_\gamma$",
        r"512 trunc $R_{\Delta Q}$",
        r"8-seed $R_\gamma$",
        r"8-seed $R_{\Delta Q}$",
    ],
    rows,
)

| tag | T1 $R_\gamma$ | T1 $R_{\Delta Q}$ | 512 trunc $R_\gamma$ | 512 trunc $R_{\Delta Q}$ | 8-seed $R_\gamma$ | 8-seed $R_{\Delta Q}$ |
| --- | --- | --- | --- | --- | --- | --- |
| `edge_s2_w1` | 1.113 | 1.156 | 1.067 | 1.093 | 1.003(10) | 1.017(10) |
| `edge_s2_w3` | 1.218 | 1.119 | 0.998 | 1.080 | 0.991(21) | 1.005(9) |
| `edge_s2_w10` | 0.999 | 1.127 | 1.111 | 1.106 | 1.009(28) | 1.022(12) |
| `edge_h05_w3` | 1.130 | 1.037 | 1.037 | 0.944 | 0.998(27) | 0.939(9) |
| `edge_h15_w3` | 1.427 | 1.110 | 1.131 | 1.024 | 1.039(15) | 0.962(11) |


In [9]:
seed = 1029
hmc = load_vector(topo_paths("hmc", EVAL_BETA, seed)["topo"])
base = load_vector(topo_paths("base", EVAL_BETA, seed)["topo"])
h15 = load_vector(topo_paths("edge_h15_w3", EVAL_BETA, seed)["topo"])
fig, ax = default_plot()
ax.plot(hmc, color=color_ls[2], lw=0.7, label="HMC")
ax.plot(base, color=color_ls[0], lw=0.8, label="production base")
ax.plot(h15, color=color_ls[3], lw=0.8, label=r"edge_h15_w3")
ax.set_xlabel(r"trajectory $t$", **fs_p)
ax.set_ylabel(r"$Q(t)$", **fs_p)
ax.legend(frameon=False, **fs_small_p)
fig.savefig(PLOT_DIR / "edge_weight_q_t_seed1029.pdf", bbox_inches="tight")
fig

## $\beta=5$ seed-1029 probe

A single-seed check at a more frozen coupling, using the same $\beta=3$ checkpoints. This is not an eight-seed claim. Protocol was to expand extra betas only for tags that remained competitive at $\beta=3$; none did.

In [10]:
probe_rows = []
for tag in ["edge_s2_w1", "edge_s2_w3", "edge_h15_w3"]:
    probe = collect_tag(tag, beta=5.0, seeds=[1029])
    row = probe["per_seed"][0]
    probe_rows.append(
        [
            f"`{tag}`",
            f"{row['accept']:.3f}",
            f"{row['step']:.3f}",
            f"{row['r_gamma_vs_base']:.3f}",
            f"{row['r_deltaq_vs_base']:.3f}",
            f"{row['r_gamma_vs_hmc']:.3f}",
            f"{row['r_deltaq_vs_hmc']:.3f}",
        ]
    )
show_table(
    ["tag", "accept", "step", r"$R_\gamma$ vs base", r"$R_{\Delta Q}$ vs base", r"$R_\gamma$ vs HMC", r"$R_{\Delta Q}$ vs HMC"],
    probe_rows,
)

| tag | accept | step | $R_\gamma$ vs base | $R_{\Delta Q}$ vs base | $R_\gamma$ vs HMC | $R_{\Delta Q}$ vs HMC |
| --- | --- | --- | --- | --- | --- | --- |
| `edge_s2_w1` | 0.658 | 0.350 | 1.058 | 1.028 | 1.444 | 1.632 |
| `edge_s2_w3` | 0.688 | 0.350 | 0.903 | 0.935 | 1.233 | 1.485 |
| `edge_h15_w3` | 0.577 | 0.350 | 1.094 | 1.009 | 1.493 | 1.603 |


## Bottom line

The large-angle / edge-link weight does **not** give a reproducible topology improvement over production `base` at the training beta.

- Smooth $\alpha=1,3,10$: $R_\gamma$ consistent with `base`; $R_{\Delta Q}$ excesses of $0$–$2\%$. That is seed scatter, not the T1 $20\%$–$43\%$ ranking.
- `edge_h15_w3`: $+2.6\sigma$ in $R_\gamma$, $-3.5\sigma$ in $R_{\Delta Q}$. Fails the both-metrics rule.
- `edge_h05_w3`: $R_{\Delta Q}$ is worse at $\sim 7\sigma$.

Extra $\beta=4,5$ eight-seed FT-HMC is not justified. Keep `edge_weight=0` as the 2du1 training default.